# **LAB 5: RNN FOR PART–OF–SPEECH TAGGING**

## **Task 1 - Load & Preprocess UD .conllu Files**

In [1]:
# -----------------------
# 1. Load .conllu file
# -----------------------
def load_conllu(file_path):
    sentences = []
    current_sentence = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:  # end of one sentence
                if current_sentence:
                    sentences.append(current_sentence)
                    current_sentence = []
                continue

            if line.startswith("#"):  # metadata
                continue

            parts = line.split("\t")
            if "-" in parts[0] or "." in parts[0]:
                continue

            word = parts[1]
            tag = parts[3]
            current_sentence.append((word, tag))

    if current_sentence:
        sentences.append(current_sentence)

    return sentences


In [2]:
# Load train + dev sets
train_file = "../data/UD_English-EWT/en_ewt-ud-train.conllu"
dev_file   = "../data/UD_English-EWT/en_ewt-ud-dev.conllu"

train_data = load_conllu(train_file)
dev_data   = load_conllu(dev_file)

print("Train sentences:", len(train_data))
print("Dev sentences:", len(dev_data))
print(train_data[0][:10])


Train sentences: 12544
Dev sentences: 2001
[('Al', 'PROPN'), ('-', 'PUNCT'), ('Zaman', 'PROPN'), (':', 'PUNCT'), ('American', 'ADJ'), ('forces', 'NOUN'), ('killed', 'VERB'), ('Shaikh', 'PROPN'), ('Abdullah', 'PROPN'), ('al', 'PROPN')]


## **Task 2 — Build Vocabulary**

In [3]:
# Special tokens
UNK = "<UNK>"
PAD = "<PAD>"

def build_vocab(dataset):
    word_to_ix = {PAD: 0, UNK: 1}
    tag_to_ix  = {PAD: 0}

    for sent in dataset:
        for word, tag in sent:
            if word not in word_to_ix:
                word_to_ix[word] = len(word_to_ix)

            if tag not in tag_to_ix:
                tag_to_ix[tag] = len(tag_to_ix)

    return word_to_ix, tag_to_ix


word_to_ix, tag_to_ix = build_vocab(train_data)

print("Word vocab size:", len(word_to_ix))
print("Tag vocab size:", len(tag_to_ix))


Word vocab size: 19675
Tag vocab size: 18


## **Task 3 — Dataset + DataLoader (Padding)**

In [5]:
from torch.utils.data import Dataset, DataLoader

class POSDataset(Dataset):
    def __init__(self, sentences, word_to_ix, tag_to_ix):
        self.sentences = sentences
        self.word_to_ix = word_to_ix
        self.tag_to_ix = tag_to_ix

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        words, tags = zip(*self.sentences[idx])
        word_ids = [self.word_to_ix.get(w, self.word_to_ix[UNK]) for w in words]
        tag_ids  = [self.tag_to_ix[t] for t in tags]
        return torch.tensor(word_ids), torch.tensor(tag_ids)


def collate_fn(batch):
    word_seqs = [item[0] for item in batch]
    tag_seqs  = [item[1] for item in batch]

    padded_words = pad_sequence(word_seqs, batch_first=True, padding_value=word_to_ix[PAD])
    padded_tags  = pad_sequence(tag_seqs, batch_first=True, padding_value=tag_to_ix[PAD])

    lengths = torch.tensor([len(x) for x in word_seqs])

    return padded_words, padded_tags, lengths


In [6]:
# Create DataLoaders
train_dataset = POSDataset(train_data, word_to_ix, tag_to_ix)
dev_dataset   = POSDataset(dev_data, word_to_ix, tag_to_ix)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


## **Task 4 — Simple RNN Model**

In [10]:
import torch 
import torch.nn as nn
class SimpleRNNForTokenClassification(nn.Module):
    def __init__(self, vocab_size, tagset_size, emb_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=word_to_ix[PAD])
        self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, x, lengths):
        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, hidden = self.rnn(packed)

        rnn_out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)

        logits = self.fc(rnn_out)
        return logits


In [11]:
# Train Model
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SimpleRNNForTokenClassification(len(word_to_ix), len(tag_to_ix)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=tag_to_ix[PAD])


In [13]:
from torch.nn.utils.rnn import pad_sequence

# Training Loop
def train_model(epochs=5):
    model.train()

    for ep in range(epochs):
        total_loss = 0

        for words, tags, lengths in train_loader:
            words, tags, lengths = words.to(device), tags.to(device), lengths.to(device)

            optimizer.zero_grad()
            logits = model(words, lengths)

            loss = criterion(logits.view(-1, logits.size(-1)), tags.view(-1))
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {ep+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

train_model(epochs=5)


Epoch 1/5 - Loss: 1.0720
Epoch 2/5 - Loss: 0.5835
Epoch 3/5 - Loss: 0.4306
Epoch 4/5 - Loss: 0.3374
Epoch 5/5 - Loss: 0.2714


## **Task 5 - Evaluate Accuracy**

In [17]:
def evaluate(loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for words, tags, lengths in loader:
            words, tags = words.to(device), tags.to(device)
            logits = model(words, lengths)
            preds = logits.argmax(dim=-1)

            mask = tags != tag_to_ix[PAD]
            correct += (preds[mask] == tags[mask]).sum().item()
            total += mask.sum().item()

    return correct / total


train_acc = evaluate(train_loader)
dev_acc   = evaluate(dev_loader)

print("Train accuracy:", train_acc)
print("Dev accuracy:", dev_acc)


Train accuracy: 0.9307497910322274
Dev accuracy: 0.8552626346972046


In [15]:
def predict_sentence(sentence):
    model.eval()
    tokens = sentence.split()
    ids = [word_to_ix.get(w, word_to_ix[UNK]) for w in tokens]
    x = torch.tensor(ids).unsqueeze(0).to(device)
    lengths = torch.tensor([len(ids)])

    with torch.no_grad():
        logits = model(x, lengths)
        preds = logits.argmax(-1).squeeze().tolist()

    ix_to_tag = {v:k for k,v in tag_to_ix.items()}

    return list(zip(tokens, [ix_to_tag[p] for p in preds]))


In [ ]:
predict_sentence("I love NLP")

[('I', 'PRON'), ('love', 'VERB'), ('NLP', 'VERB')]

In [19]:
predict_sentence("They will travel to Japan tomorrow")

[('They', 'PRON'),
 ('will', 'AUX'),
 ('travel', 'VERB'),
 ('to', 'ADP'),
 ('Japan', 'PROPN'),
 ('tomorrow', 'NOUN')]

In [20]:
predict_sentence("This movie is absolutely fantastic")

[('This', 'DET'),
 ('movie', 'NOUN'),
 ('is', 'AUX'),
 ('absolutely', 'ADV'),
 ('fantastic', 'ADJ')]

In [21]:
predict_sentence("Students are studying in the library")

[('Students', 'VERB'),
 ('are', 'AUX'),
 ('studying', 'VERB'),
 ('in', 'ADP'),
 ('the', 'DET'),
 ('library', 'NOUN')]